In [46]:
import re
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.alert import Alert
from selenium.common.exceptions import UnexpectedAlertPresentException, NoSuchElementException

In [47]:
# 웹 드라이버 설정
options = webdriver.ChromeOptions()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

# 버전에 상관없이 설치된 크롬 브라우저 사용
driver = webdriver.Chrome()
driver.implicitly_wait(3)

In [48]:
# HTML 파싱 및 콘텐츠 추출 함수
def get_content(driver):
    time.sleep(1)  # 페이지 로딩 대기

    # 🔍 1. 팝업(Alert) 감지 및 자동 닫기
    try:
        alert = Alert(driver)
        alert.dismiss()  # 팝업 닫기
        print("⚠️ 경고 팝업 감지 - 자동 닫음.")
        time.sleep(1)
    except Exception:
        pass

    # 🔍 2. HTML 파싱
    soup = BeautifulSoup(driver.page_source, "html.parser")

    # 🔍 3. `iframe` 전환
    try:
        iframe = driver.find_element(By.CSS_SELECTOR, "iframe#mainFrame")
        driver.switch_to.frame(iframe)
        time.sleep(1)
        soup = BeautifulSoup(driver.page_source, "html.parser")
    except NoSuchElementException:
        pass

    try:
        date = soup.select_one('span.se_publishDate, span.post-date').get_text().strip()
    except:
        date = ''

    try:
        title = soup.select_one('div.se-title-text, h3#title_area, h3.post-title').get_text().strip()
    except:
        title = ''

    try:
        content = soup.select_one('div.se-main-container, div.tt_article_useless_p_margin').get_text()
    except:
        content = ''

    link = driver.current_url
    return [title, content, date, link]

In [23]:
urls = pd.read_csv("무선스틱청소기url_2024.csv")
naver_urls = urls['link'].tolist()

In [ ]:
naver_urls

In [ ]:
# 크롤링 실행
results = []
count = 0

In [49]:
# 중간에 멈추면 naver_urls[멈춘count값:]으로 다시 실행
for url in naver_urls[3589:]:
    driver.get(url)
    time.sleep(1)

    blog_data = get_content(driver)
    results.append(blog_data)
    count += 1

    if count % 100 == 0:
        print(f"{count}개 크롤링 완료")

3600개 크롤링 완료
3700개 크롤링 완료
3800개 크롤링 완료
3900개 크롤링 완료
4000개 크롤링 완료
4100개 크롤링 완료
4200개 크롤링 완료
4300개 크롤링 완료
⚠️ 경고 팝업 감지 - 자동 닫음.
4400개 크롤링 완료
4500개 크롤링 완료
4600개 크롤링 완료
4700개 크롤링 완료
4800개 크롤링 완료
4900개 크롤링 완료
5000개 크롤링 완료
5100개 크롤링 완료
5200개 크롤링 완료
5300개 크롤링 완료
5400개 크롤링 완료


In [50]:
# DataFrame 생성 및 저장
df = pd.DataFrame(results)
df["platform"] = "blog"
df.columns = ['title', 'content', 'date', 'link', 'platform']
df.to_csv('무선스틱청소기_2024.csv', index=False, encoding='utf-8-sig')

In [43]:
count

3588

In [28]:
naver_urls[1933]

'https://blog.naver.com/zestgo/223548365608'

In [51]:
df.head()

,title,content,date,link,platform
0,삶의질업템강추 삼성전자 싸이클론 진공청소기 VC33M2110LP 강력한 청소력,삶의질업템강추 삼성전자 싸이클론 진공청소기 VC33M2110LP 강력한 청소력 오...,2024. 12. 15. 14:10,https://blog.naver.com/tcgisr4624761/223694133945,blog
1,"LG 코드제로 A9S와 올인원 타워, 무선청소기의 새로운 기준을 세우다!","LG 코드제로 A9S와 올인원 타워, 무선청소기의 새로운 기준을 세우다! 안녕하세...",2024. 8. 8. 8:29,https://blog.naver.com/rlrlrlrl11/223540467021,blog
2,LG코드제로 신제품 A9 Air 가볍고 슬림한 청소기 추천합니다.,안녕하세요롯데하이마트 잠실점입니다. 오늘 소개해드릴 상품은 LG A9 Air 신제...,2024. 3. 19. 19:21,https://blog.naver.com/ohhokil/223388608954,blog
3,샤크 무선청소기 가볍고 활용 좋은 핸디청소기 에보 네오플러스,업체로부터 제품을 제공받아 작성한 리뷰입니다. 신축 아파트에 이사 와서 깨끗하게 ...,2024. 9. 13. 15:31,https://blog.naver.com/aejiniyam/223583000646,blog
4,"삼성전자 BESPOKE AI 스팀 로봇청소기의 혁명, 한국이모님 가전",(서울 =뉴스1) 한재준 기자 2024-04-03 10:00 송고 새로운 가전제품...,2024. 4. 4. 23:57,https://blog.naver.com/ever6486/223406102600,blog


In [55]:
df['link'][df['link'].duplicated(keep=False)]

2947    https://blog.naver.com/tklovessh/223707850781
2948    https://blog.naver.com/tklovessh/223707850781
Name: link, dtype: object